In [ ]:
from pathlib import Path
import json
import os
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 661
PROJECT_YEARS = list(range(2018, 2025))
PM25_THRESHOLD = 25.0
MIN_VALID_PM_HOURS = 18
MIN_VALID_COUNT_HOURS = 24
MAX_FIRE_DISTANCE_KM = 500.0
PREFILTER_DISTANCE_KM = 520.0
UPWIND_ANGLE_DEG = 45.0

# Transparent FSI research parameters. These are engineering defaults, not physical laws.
FSI_DISTANCE_TAU_KM = 100.0
FSI_AGE_HALF_LIFE_HOURS = 24.0
FSI_CONFIDENCE_WEIGHTS = {"l": 0.50, "n": 0.75, "h": 1.00}

# Optional future API acquisition. Historical Assessment 3 modelling continues to use
# the supplied 2018–2024 archive files so the evaluation period is reproducible.
USE_FIRMS_API = False
FIRMS_API_SOURCE = os.getenv("FIRMS_API_SOURCE", "VIIRS_SNPP_NRT")
FIRMS_API_BBOX = "125.5,-17.5,136.5,-7.5"  # west,south,east,north

possible_folders = [
    Path.cwd(),
    Path.cwd() / "Data Science",
    Path.home() / "Desktop" / "Data Science",
]

project_folder = None
for folder in possible_folders:
    if (folder / "ntepa_greater_darwin_hourly_2018_2024.csv").exists():
        project_folder = folder.resolve()
        break

if project_folder is None:
    raise FileNotFoundError(
        "Could not find ntepa_greater_darwin_hourly_2018_2024.csv. "
        "Place it and the seven FIRMS yearly archives (2018–2024) in the project folder."
    )

output_root = project_folder / "outputs_prt661_a3"
table_dir = output_root / "tables"
processed_dir = output_root / "processed"
for folder in [output_root, table_dir, processed_dir]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folder:", project_folder)

In [ ]:
def fetch_firms_area_api(map_key, source=FIRMS_API_SOURCE, bbox=FIRMS_API_BBOX, day_range=1, date=None):
    """Fetch a recent FIRMS Area API CSV snapshot.

    This helper is intentionally optional. It is for future automated refreshes and is
    not mixed into the historical 2018–2024 evaluation dataset by default.
    """
    if not map_key:
        raise ValueError("A FIRMS MAP_KEY is required. Store it in the FIRMS_MAP_KEY environment variable.")
    if not 1 <= int(day_range) <= 5:
        raise ValueError("FIRMS Area API day_range must be between 1 and 5.")

    base = "https://firms.modaps.eosdis.nasa.gov/api/area/csv"
    url = f"{base}/{map_key}/{source}/{bbox}/{int(day_range)}"
    if date is not None:
        url += f"/{pd.Timestamp(date).strftime('%Y-%m-%d')}"
    return pd.read_csv(url)


if USE_FIRMS_API:
    api_key = os.getenv("FIRMS_MAP_KEY")
    api_snapshot = fetch_firms_area_api(api_key)
    api_snapshot.to_csv(processed_dir / "firms_api_latest_snapshot.csv", index=False)
    print("Saved optional FIRMS API snapshot:", len(api_snapshot), "rows")

In [ ]:
ntepa_file = project_folder / "ntepa_greater_darwin_hourly_2018_2024.csv"
df = pd.read_csv(ntepa_file)
df["datetime_local"] = pd.to_datetime(df["datetime_local"], errors="coerce")
df["date"] = df["datetime_local"].dt.normalize()
df["year"] = df["datetime_local"].dt.year

required_columns = {
    "datetime_local", "station", "latitude", "longitude",
    "pm25_ug_m3", "pm10_ug_m3", "relative_humidity_pct",
    "air_temperature_c", "wind_speed_m_s", "wind_direction_deg",
    "air_pressure_hpa", "rainfall_mm",
}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"NT EPA file is missing required columns: {sorted(missing_columns)}")

quality_columns = [
    "pm25_ug_m3", "pm10_ug_m3", "relative_humidity_pct",
    "air_temperature_c", "wind_speed_m_s", "wind_direction_deg",
    "air_pressure_hpa", "rainfall_mm",
]

overview = pd.DataFrame({
    "metric": [
        "rows", "columns", "date_min", "date_max", "stations",
        "full_row_duplicates", "station_timestamp_duplicates",
    ],
    "value": [
        len(df), df.shape[1], df["datetime_local"].min(), df["datetime_local"].max(),
        ", ".join(sorted(df["station"].dropna().astype(str).unique())),
        int(df.duplicated().sum()),
        int(df.duplicated(["station", "datetime_local"]).sum()),
    ],
})
overview.to_csv(table_dir / "a3_epa_raw_overview.csv", index=False)
print("\nRaw NT EPA overview:")
print(overview.to_string(index=False))

missing_by_year_station = (
    df.groupby(["year", "station"])[quality_columns]
      .agg(lambda s: s.isna().mean() * 100)
      .round(2)
      .reset_index()
)
missing_by_year_station.to_csv(table_dir / "a3_epa_missing_pct_year_station_variable.csv", index=False)

raw_summary = df[quality_columns].describe(
    percentiles=[0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999]
).T
raw_summary.to_csv(table_dir / "a3_epa_raw_numeric_summary.csv")

In [ ]:
df_clean = df.copy()
cleaning_log = []

# Remove exact duplicates and station-timestamp duplicates before aggregation.
before_rows = len(df_clean)
df_clean = df_clean.drop_duplicates().copy()
exact_removed = before_rows - len(df_clean)
cleaning_log.append({
    "variable": "row",
    "rule": "exact duplicate row removed",
    "values_set_or_changed": int(exact_removed),
})

before_rows = len(df_clean)
df_clean = (
    df_clean.sort_values(["station", "datetime_local"])
    .drop_duplicates(["station", "datetime_local"], keep="first")
    .copy()
)
station_time_removed = before_rows - len(df_clean)
cleaning_log.append({
    "variable": "row",
    "rule": "duplicate station-timestamp removed",
    "values_set_or_changed": int(station_time_removed),
})

def log_change(variable, rule, before_nonmissing, after_nonmissing):
    cleaning_log.append({
        "variable": variable,
        "rule": rule,
        "values_set_or_changed": int(before_nonmissing - after_nonmissing),
    })


for col in ["pm25_ug_m3", "pm10_ug_m3"]:
    before = int(df_clean[col].notna().sum())
    numeric = pd.to_numeric(df_clean[col], errors="coerce")
    df_clean.loc[numeric < 0, col] = np.nan
    after = int(df_clean[col].notna().sum())
    log_change(col, "negative concentration -> NaN", before, after)

# Station-level pressure audit and harmonisation.
pressure_profile = (
    df_clean.groupby("station")["air_pressure_hpa"]
    .agg(["count", "median", "min", "max"])
    .reset_index()
)
pressure_profile["conversion_factor"] = np.where(
    pressure_profile["median"].between(0.8, 1.2, inclusive="both"),
    1013.25,
    1.0,
)
pressure_profile.to_csv(table_dir / "a3_epa_pressure_unit_audit.csv", index=False)

for row in pressure_profile.itertuples(index=False):
    if row.conversion_factor != 1.0:
        mask = df_clean["station"].eq(row.station) & df_clean["air_pressure_hpa"].notna()
        df_clean.loc[mask, "air_pressure_hpa"] = (
            pd.to_numeric(df_clean.loc[mask, "air_pressure_hpa"], errors="coerce")
            * row.conversion_factor
        )
        cleaning_log.append({
            "variable": "air_pressure_hpa",
            "rule": f"{row.station}: atmosphere-scale pressure converted to hPa x1013.25",
            "values_set_or_changed": int(mask.sum()),
        })

plausible_ranges = {
    "relative_humidity_pct": (0.0, 100.0),
    "air_temperature_c": (-20.0, 60.0),
    "wind_speed_m_s": (0.0, 80.0),
    "air_pressure_hpa": (850.0, 1100.0),
}
for col, (low, high) in plausible_ranges.items():
    before = int(df_clean[col].notna().sum())
    numeric = pd.to_numeric(df_clean[col], errors="coerce")
    valid = numeric.between(low, high, inclusive="both")
    df_clean.loc[df_clean[col].notna() & ~valid, col] = np.nan
    after = int(df_clean[col].notna().sum())
    log_change(col, f"outside [{low}, {high}] -> NaN", before, after)

before = int(df_clean["wind_direction_deg"].notna().sum())
wind_numeric = pd.to_numeric(df_clean["wind_direction_deg"], errors="coerce")
valid_wind_dir = wind_numeric.between(0.0, 360.0, inclusive="left")
df_clean.loc[df_clean["wind_direction_deg"].notna() & ~valid_wind_dir, "wind_direction_deg"] = np.nan
after = int(df_clean["wind_direction_deg"].notna().sum())
log_change("wind_direction_deg", "outside [0, 360) -> NaN", before, after)

# Station-specific rainfall behaviour audit.
rain_profile_rows = []
for station, group in df_clean.sort_values("datetime_local").groupby("station"):
    s = pd.to_numeric(group["rainfall_mm"], errors="coerce")
    nonmissing = s.dropna()
    rain_profile_rows.append({
        "station": station,
        "nonmissing_pct": 100 * s.notna().mean(),
        "median_raw": nonmissing.median() if not nonmissing.empty else np.nan,
        "p95_raw": nonmissing.quantile(0.95) if not nonmissing.empty else np.nan,
        "zero_fraction_raw": nonmissing.eq(0).mean() if not nonmissing.empty else np.nan,
        "n_unique_raw": nonmissing.nunique(),
    })

rain_profile = pd.DataFrame(rain_profile_rows)
rain_profile["cumulative_counter_flag"] = (
    (rain_profile["median_raw"] > 100)
    & (rain_profile["zero_fraction_raw"].fillna(1.0) < 0.20)
)
rain_profile.to_csv(table_dir / "a3_epa_rainfall_station_behaviour_audit.csv", index=False)

df_clean = df_clean.sort_values(["station", "datetime_local"]).copy()
df_clean["rainfall_hourly_mm"] = np.nan
cumulative_stations = set(rain_profile.loc[rain_profile["cumulative_counter_flag"], "station"])

for station, idx in df_clean.groupby("station").groups.items():
    g = df_clean.loc[idx].sort_values("datetime_local")
    raw_rain = pd.to_numeric(g["rainfall_mm"], errors="coerce")
    if station in cumulative_stations:
        diffs = raw_rain.diff()
        hour_gap = g["datetime_local"].diff().dt.total_seconds().div(3600)
        derived = diffs.where(hour_gap.eq(1))
        derived = derived.where(derived >= 0)
        derived = derived.where(derived <= 300)
        df_clean.loc[g.index, "rainfall_hourly_mm"] = derived
    else:
        df_clean.loc[g.index, "rainfall_hourly_mm"] = raw_rain.where(raw_rain.between(0, 300))

cleaning_log_df = pd.DataFrame(cleaning_log)
cleaning_log_df.to_csv(table_dir / "a3_epa_cleaning_log.csv", index=False)

cleaned_ntepa_file = processed_dir / "NT_EPA_Air_Quality_Cleaned_2018_2024_A3.csv"
df_clean.to_csv(cleaned_ntepa_file, index=False)
print("\nCleaned NT EPA shape:", df_clean.shape)

In [ ]:
firms_files = sorted([
    p for p in project_folder.glob("fire_archive_SV-C2_*")
    if p.is_file() and p.suffix.lower() in {".csv", ".txt"}
])
if not firms_files:
    raise FileNotFoundError("No FIRMS yearly archives found matching fire_archive_SV-C2_* (.csv/.txt).")

print("\nFIRMS files found:", len(firms_files))
for file in firms_files:
    print(" -", file.name)

station_coordinates = {
    "Palmerston": (-12.507753, 130.948253),
    "Stokes Hill": (-12.459991, 130.847847),
    "Winnellie": (-12.424017, 130.893346),
}

def haversine_distance(lat, lon, station_lat, station_lon):
    earth_radius = 6371.0088
    lat1 = np.radians(np.asarray(lat, dtype=float))
    lon1 = np.radians(np.asarray(lon, dtype=float))
    lat2 = np.radians(station_lat)
    lon2 = np.radians(station_lon)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )
    a = np.clip(a, 0.0, 1.0)
    return 2 * earth_radius * np.arcsin(np.sqrt(a))


def initial_bearing_deg(source_lat, source_lon, target_lat, target_lon):
    """Initial bearing from source (fire) to target (monitoring station), 0–360 degrees."""
    lat1 = np.radians(np.asarray(source_lat, dtype=float))
    lon1 = np.radians(np.asarray(source_lon, dtype=float))
    lat2 = np.radians(float(target_lat))
    lon2 = np.radians(float(target_lon))
    dlon = lon2 - lon1
    y = np.sin(dlon) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(y, x)) + 360.0) % 360.0

In [ ]:
processed_chunks = []
firms_audit_rows = []
raw_utc_dates = set()

for file in firms_files:
    file_rows = 0
    kept_rows = 0
    min_date = None
    max_date = None
    for chunk in pd.read_csv(file, chunksize=500_000):
        file_rows += len(chunk)
        required_firms = {"latitude", "longitude", "acq_date", "acq_time", "frp"}
        missing_firms = required_firms.difference(chunk.columns)
        if missing_firms:
            raise ValueError(f"{file.name} is missing required FIRMS columns: {sorted(missing_firms)}")

        parsed_dates = pd.to_datetime(chunk["acq_date"], errors="coerce")
        raw_utc_dates.update(parsed_dates.dropna().dt.normalize().tolist())
        cmin, cmax = parsed_dates.min(), parsed_dates.max()
        if pd.notna(cmin):
            min_date = cmin if min_date is None or cmin < min_date else min_date
        if pd.notna(cmax):
            max_date = cmax if max_date is None or cmax > max_date else max_date

        for col in ["latitude", "longitude", "frp", "brightness", "bright_ti4", "bright_ti5"]:
            if col in chunk.columns:
                chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

        valid_coord = (
            chunk["latitude"].between(-90, 90)
            & chunk["longitude"].between(-180, 180)
            & parsed_dates.notna()
        )
        chunk = chunk.loc[valid_coord].copy()
        if chunk.empty:
            continue

        # Geographic prefilter around the Top End before exact station distance.
        chunk = chunk[
            chunk["latitude"].between(-17.5, -7.5)
            & chunk["longitude"].between(125.5, 136.5)
        ].copy()
        if chunk.empty:
            continue

        station_distances = []
        for station_lat, station_lon in station_coordinates.values():
            station_distances.append(
                haversine_distance(
                    chunk["latitude"].to_numpy(),
                    chunk["longitude"].to_numpy(),
                    station_lat,
                    station_lon,
                )
            )
        chunk["nearest_distance_km"] = np.min(np.vstack(station_distances), axis=0)
        chunk = chunk[chunk["nearest_distance_km"] <= PREFILTER_DISTANCE_KM].copy()
        if chunk.empty:
            continue

        if "frp" in chunk.columns:
            chunk.loc[chunk["frp"] < 0, "frp"] = np.nan

        if "acq_time" not in chunk.columns:
            raise ValueError(f"{file.name} is missing acq_time")
        chunk["acq_time_str"] = (
            chunk["acq_time"].astype(str).str.replace(".0", "", regex=False).str.zfill(4)
        )
        chunk["datetime_utc"] = pd.to_datetime(
            chunk["acq_date"].astype(str) + " " + chunk["acq_time_str"],
            format="%Y-%m-%d %H%M",
            utc=True,
            errors="coerce",
        )
        chunk["datetime_darwin"] = chunk["datetime_utc"].dt.tz_convert("Australia/Darwin")
        chunk["datetime_darwin_naive"] = chunk["datetime_darwin"].dt.tz_localize(None)
        chunk["date_darwin"] = chunk["datetime_darwin_naive"].dt.normalize()
        kept_rows += len(chunk)
        processed_chunks.append(chunk)

    filename_year = None
    year_match = re.search(r"(20\d{2})", file.stem)
    if year_match:
        filename_year = int(year_match.group(1))
    firms_audit_rows.append({
        "file": file.name,
        "filename_year": filename_year,
        "raw_rows": file_rows,
        "prefiltered_rows": kept_rows,
        "min_acq_date": min_date,
        "max_acq_date": max_date,
    })

if not processed_chunks:
    raise ValueError("No FIRMS detections remained after the Darwin-region prefilter.")

firms_darwin = pd.concat(processed_chunks, ignore_index=True)
del processed_chunks

firms_file_audit = pd.DataFrame(firms_audit_rows)
firms_file_audit.to_csv(table_dir / "a3_firms_file_audit.csv", index=False)

# Verify actual archive years from acquisition dates, not only filenames.
actual_firms_years = sorted(
    pd.to_datetime(firms_darwin["acq_date"], errors="coerce").dropna().dt.year.unique().tolist()
)
year_coverage = pd.DataFrame({
    "year": PROJECT_YEARS,
    "present_in_firms_archives": [int(y in actual_firms_years) for y in PROJECT_YEARS],
})
year_coverage.to_csv(table_dir / "a3_firms_year_coverage_2018_2024.csv", index=False)
missing_years = [y for y in PROJECT_YEARS if y not in actual_firms_years]
if missing_years:
    raise ValueError(
        "Assessment 3 requires FIRMS 2018–2024 coverage, but these years are missing: "
        f"{missing_years}. Check the seven supplied fire_archive files."
    )
print("Verified FIRMS years:", actual_firms_years)

# Deduplicate overlapping archive detections.
dedup_candidates = [
    "latitude", "longitude", "brightness", "bright_ti4", "acq_date", "acq_time",
    "satellite", "instrument", "confidence", "frp", "daynight", "type",
]
dedup_key = [c for c in dedup_candidates if c in firms_darwin.columns]
duplicates_before = int(firms_darwin.duplicated(subset=dedup_key).sum())
firms_darwin = firms_darwin.drop_duplicates(subset=dedup_key, keep="first").copy()

if "type" in firms_darwin.columns:
    firms_fire = firms_darwin[firms_darwin["type"].eq(0)].copy()
else:
    print("WARNING: FIRMS 'type' column absent; all retained thermal detections are used.")
    firms_fire = firms_darwin.copy()

print("Darwin-region detections after deduplication:", len(firms_darwin))
print("Duplicate detections removed:", duplicates_before)
print("Vegetation-fire detections used:", len(firms_fire))

# Station-specific distance and fire-to-station bearing.
for station_name, (station_lat, station_lon) in station_coordinates.items():
    station_key = station_name.lower().replace(" ", "_")
    firms_fire[f"distance_{station_key}_km"] = haversine_distance(
        firms_fire["latitude"].to_numpy(),
        firms_fire["longitude"].to_numpy(),
        station_lat,
        station_lon,
    )
    firms_fire[f"bearing_fire_to_{station_key}_deg"] = initial_bearing_deg(
        firms_fire["latitude"].to_numpy(),
        firms_fire["longitude"].to_numpy(),
        station_lat,
        station_lon,
    )

In [ ]:
distance_bins = [0, 25, 50, 100, 200, 500]
distance_labels = ["0_25km", "25_50km", "50_100km", "100_200km", "200_500km"]

def create_station_daily_basic_features(data, station_name):
    station_key = station_name.lower().replace(" ", "_")
    distance_col = f"distance_{station_key}_km"
    station_data = data[data[distance_col] <= MAX_FIRE_DISTANCE_KM].copy()
    if station_data.empty:
        return pd.DataFrame(columns=["date_darwin", "station"])

    station_data["distance_band"] = pd.cut(
        station_data[distance_col],
        bins=distance_bins,
        labels=distance_labels,
        include_lowest=True,
        right=True,
    )

    band_features = (
        station_data.groupby(["date_darwin", "distance_band"], observed=True)
        .agg(
            fire_count=("latitude", "size"),
            frp_sum=("frp", "sum"),
            frp_mean=("frp", "mean"),
            frp_max=("frp", "max"),
        )
        .unstack()
    )
    expected_columns = pd.MultiIndex.from_product(
        [["fire_count", "frp_sum", "frp_mean", "frp_max"], distance_labels]
    )
    band_features = band_features.reindex(columns=expected_columns)
    band_features.columns = [f"{metric}_{band}" for metric, band in band_features.columns]
    band_features = band_features.reset_index()

    overall = (
        station_data.groupby("date_darwin")
        .agg(
            fire_count_0_500km=("latitude", "size"),
            frp_sum_0_500km=("frp", "sum"),
            frp_mean_0_500km=("frp", "mean"),
            frp_max_0_500km=("frp", "max"),
            nearest_fire_km=(distance_col, "min"),
        )
        .reset_index()
    )
    out = band_features.merge(overall, on="date_darwin", how="outer", validate="one_to_one")
    out["station"] = station_name
    return out

station_daily_fire = pd.concat(
    [create_station_daily_basic_features(firms_fire, station) for station in station_coordinates],
    ignore_index=True,
)

all_dates = pd.date_range("2018-01-01", "2024-12-31", freq="D")
station_calendar = (
    pd.MultiIndex.from_product(
        [all_dates, list(station_coordinates.keys())],
        names=["date_darwin", "station"],
    )
    .to_frame(index=False)
)

station_fire_complete = station_calendar.merge(
    station_daily_fire,
    on=["date_darwin", "station"],
    how="left",
    validate="one_to_one",
)

coverage = pd.DataFrame({"date_darwin": all_dates})
coverage["firms_local_day_complete"] = coverage["date_darwin"].apply(
    lambda date: int(
        date.normalize() in raw_utc_dates
        and (date - pd.Timedelta(days=1)).normalize() in raw_utc_dates
    )
)
coverage["firms_48h_complete"] = (
    coverage["firms_local_day_complete"].eq(1)
    & coverage["firms_local_day_complete"].shift(1).fillna(0).eq(1)
).astype(int)
station_fire_complete = station_fire_complete.merge(
    coverage,
    on="date_darwin",
    how="left",
    validate="many_to_one",
)

count_sum_cols = [
    c for c in station_fire_complete.columns
    if c.startswith("fire_count_") or c.startswith("frp_sum_")
]
fire_feature_cols = [
    c for c in station_fire_complete.columns
    if c.startswith("fire_count_")
    or c.startswith("frp_sum_")
    or c.startswith("frp_mean_")
    or c.startswith("frp_max_")
    or c == "nearest_fire_km"
]
complete_firms_days = station_fire_complete["firms_local_day_complete"].eq(1)
station_fire_complete.loc[complete_firms_days, count_sum_cols] = (
    station_fire_complete.loc[complete_firms_days, count_sum_cols].fillna(0)
)
station_fire_complete.loc[~complete_firms_days, fire_feature_cols] = np.nan

station_fire_complete["fire_day"] = pd.NA
station_fire_complete.loc[complete_firms_days, "fire_day"] = (
    station_fire_complete.loc[complete_firms_days, "fire_count_0_500km"] > 0
).astype(int)
station_fire_complete["fire_day"] = station_fire_complete["fire_day"].astype("Int64")

In [ ]:
df_clean["date"] = df_clean["datetime_local"].dt.normalize()
df_clean = df_clean.sort_values(["station", "datetime_local"]).copy()

# Circular wind components.
wind_rad = np.deg2rad(df_clean["wind_direction_deg"])
df_clean["wind_dir_sin"] = np.sin(wind_rad)
df_clean["wind_dir_cos"] = np.cos(wind_rad)

daily_pm25 = (
    df_clean.groupby(["station", "date"])
    .agg(
        pm25_mean=("pm25_ug_m3", "mean"),
        pm25_min=("pm25_ug_m3", "min"),
        pm25_max=("pm25_ug_m3", "max"),
        pm25_std=("pm25_ug_m3", "std"),
        pm25_valid_hours=("pm25_ug_m3", "count"),
        hours_pm25_ge_25=("pm25_ug_m3", lambda x: (x >= PM25_THRESHOLD).sum()),
    )
    .reset_index()
)
daily_pm25.loc[
    daily_pm25["pm25_valid_hours"] < MIN_VALID_PM_HOURS,
    ["pm25_mean", "pm25_min", "pm25_max", "pm25_std"],
] = np.nan
daily_pm25.loc[
    daily_pm25["pm25_valid_hours"] < MIN_VALID_COUNT_HOURS,
    "hours_pm25_ge_25",
] = np.nan

daily_pm10 = (
    df_clean.groupby(["station", "date"])
    .agg(
        pm10_mean=("pm10_ug_m3", "mean"),
        pm10_min=("pm10_ug_m3", "min"),
        pm10_max=("pm10_ug_m3", "max"),
        pm10_valid_hours=("pm10_ug_m3", "count"),
    )
    .reset_index()
)
daily_pm10.loc[
    daily_pm10["pm10_valid_hours"] < MIN_VALID_PM_HOURS,
    ["pm10_mean", "pm10_min", "pm10_max"],
] = np.nan

# First/last pressure are used for a within-day pressure-change feature.
def first_valid(s):
    z = s.dropna()
    return z.iloc[0] if len(z) else np.nan


def last_valid(s):
    z = s.dropna()
    return z.iloc[-1] if len(z) else np.nan

daily_weather = (
    df_clean.groupby(["station", "date"])
    .agg(
        humidity_mean=("relative_humidity_pct", "mean"),
        humidity_min=("relative_humidity_pct", "min"),
        humidity_max=("relative_humidity_pct", "max"),
        temperature_mean=("air_temperature_c", "mean"),
        temperature_min=("air_temperature_c", "min"),
        temperature_max=("air_temperature_c", "max"),
        wind_speed_mean=("wind_speed_m_s", "mean"),
        wind_speed_max=("wind_speed_m_s", "max"),
        pressure_mean=("air_pressure_hpa", "mean"),
        pressure_first=("air_pressure_hpa", first_valid),
        pressure_last=("air_pressure_hpa", last_valid),
        rainfall_total=("rainfall_hourly_mm", lambda x: x.sum(min_count=1)),
        wind_dir_sin_mean=("wind_dir_sin", "mean"),
        wind_dir_cos_mean=("wind_dir_cos", "mean"),
    )
    .reset_index()
)
daily_weather["pressure_change"] = daily_weather["pressure_last"] - daily_weather["pressure_first"]
daily_weather = daily_weather.drop(columns=["pressure_first", "pressure_last"])

daily_weather_coverage = (
    df_clean.groupby(["station", "date"])
    .agg(
        humidity_hours=("relative_humidity_pct", "count"),
        temperature_hours=("air_temperature_c", "count"),
        wind_speed_hours=("wind_speed_m_s", "count"),
        pressure_hours=("air_pressure_hpa", "count"),
        wind_direction_hours=("wind_direction_deg", "count"),
    )
    .reset_index()
)

ntepa_daily = (
    daily_pm25
    .merge(daily_pm10, on=["station", "date"], how="left", validate="one_to_one")
    .merge(daily_weather, on=["station", "date"], how="left", validate="one_to_one")
    .merge(daily_weather_coverage, on=["station", "date"], how="left", validate="one_to_one")
)

# Coverage rules for daily weather summaries.
weather_rules = {
    "humidity_hours": ["humidity_mean", "humidity_min", "humidity_max"],
    "temperature_hours": ["temperature_mean", "temperature_min", "temperature_max"],
    "wind_speed_hours": ["wind_speed_mean", "wind_speed_max"],
    "pressure_hours": ["pressure_mean", "pressure_change"],
    "wind_direction_hours": ["wind_dir_sin_mean", "wind_dir_cos_mean"],
}
for coverage_col, value_cols in weather_rules.items():
    ntepa_daily.loc[ntepa_daily[coverage_col] < MIN_VALID_PM_HOURS, value_cols] = np.nan

ntepa_daily["pm25_coverage_ratio"] = ntepa_daily["pm25_valid_hours"] / 24.0
ntepa_daily["pm10_coverage_ratio"] = ntepa_daily["pm10_valid_hours"] / 24.0
ntepa_daily["humidity_coverage_ratio"] = ntepa_daily["humidity_hours"] / 24.0
ntepa_daily["temperature_coverage_ratio"] = ntepa_daily["temperature_hours"] / 24.0
ntepa_daily["wind_speed_coverage_ratio"] = ntepa_daily["wind_speed_hours"] / 24.0
ntepa_daily["pressure_coverage_ratio"] = ntepa_daily["pressure_hours"] / 24.0
ntepa_daily["wind_direction_coverage_ratio"] = ntepa_daily["wind_direction_hours"] / 24.0

ntepa_daily["year"] = ntepa_daily["date"].dt.year
ntepa_daily["month"] = ntepa_daily["date"].dt.month

In [ ]:
def confidence_weight(series):
    """Convert VIIRS confidence to a transparent research weight.

    Supports categorical l/n/h and numeric confidence fields. The mapping is recorded in
    the feature manifest and should be treated as a sensitivity parameter.
    """
    s = series.astype(str).str.strip().str.lower()
    mapped = s.map(FSI_CONFIDENCE_WEIGHTS)
    numeric = pd.to_numeric(series, errors="coerce")
    numeric_weight = (numeric / 100.0).clip(lower=0.0, upper=1.0)
    return mapped.fillna(numeric_weight).fillna(1.0)

In [ ]:
def build_advanced_fire_features_for_station(fire_data, station_name, weather_daily):
    station_key = station_name.lower().replace(" ", "_")
    distance_col = f"distance_{station_key}_km"
    bearing_col = f"bearing_fire_to_{station_key}_deg"

    cols = [
        "datetime_darwin_naive", "date_darwin", "latitude", "longitude",
        distance_col, bearing_col,
    ]
    for optional in ["frp", "confidence", "brightness", "bright_ti4", "daynight"]:
        if optional in fire_data.columns:
            cols.append(optional)

    station_data = fire_data.loc[fire_data[distance_col] <= MAX_FIRE_DISTANCE_KM, cols].copy()
    if station_data.empty:
        return pd.DataFrame(columns=["date_darwin", "station"])

    if "frp" not in station_data.columns:
        station_data["frp"] = np.nan
    if "confidence" not in station_data.columns:
        station_data["confidence"] = np.nan

    base = station_data.rename(columns={distance_col: "distance_km", bearing_col: "bearing_fire_to_station_deg"})

    # Each fire can influence its own predictor day (0–24 h old at end-of-day cut-off)
    # and the following predictor day (24–48 h old). This creates a transparent 48 h window.
    current = base.copy()
    current["predictor_date"] = current["date_darwin"]
    previous = base.copy()
    previous["predictor_date"] = previous["date_darwin"] + pd.Timedelta(days=1)
    events = pd.concat([current, previous], ignore_index=True)
    events = events[events["predictor_date"].between(all_dates.min(), all_dates.max())].copy()

    events["prediction_cutoff"] = events["predictor_date"] + pd.Timedelta(days=1)
    events["fire_age_hours"] = (
        events["prediction_cutoff"] - events["datetime_darwin_naive"]
    ).dt.total_seconds().div(3600.0)
    events = events[events["fire_age_hours"].between(0, 48, inclusive="both")].copy()

    weather_station = (
        weather_daily.loc[weather_daily["station"].eq(station_name), [
            "date", "wind_speed_mean", "wind_dir_sin_mean", "wind_dir_cos_mean"
        ]]
        .rename(columns={"date": "predictor_date"})
    )
    events = events.merge(weather_station, on="predictor_date", how="left", validate="many_to_one")

    events["wind_from_deg"] = (
        np.degrees(np.arctan2(events["wind_dir_sin_mean"], events["wind_dir_cos_mean"])) + 360.0
    ) % 360.0
    # Meteorological wind direction is where wind comes FROM; smoke transport is toward +180°.
    events["smoke_transport_deg"] = (events["wind_from_deg"] + 180.0) % 360.0
    events["angular_difference_deg"] = np.abs(
        ((events["smoke_transport_deg"] - events["bearing_fire_to_station_deg"] + 180.0) % 360.0) - 180.0
    )
    events["wind_alignment"] = np.maximum(
        0.0, np.cos(np.radians(events["angular_difference_deg"]))
    )
    events.loc[events["smoke_transport_deg"].isna(), "wind_alignment"] = np.nan

    events["upwind_flag"] = np.where(
        events["wind_alignment"].isna(),
        np.nan,
        (events["angular_difference_deg"] <= UPWIND_ANGLE_DEG).astype(float),
    )

    # Age windows requested in the Assessment 3 strategy.
    events["count_le_6h"] = (events["fire_age_hours"] <= 6).astype(int)
    events["count_le_12h"] = (events["fire_age_hours"] <= 12).astype(int)
    events["count_le_24h"] = (events["fire_age_hours"] <= 24).astype(int)
    events["count_24_48h"] = ((events["fire_age_hours"] > 24) & (events["fire_age_hours"] <= 48)).astype(int)

    for label, mask in {
        "le_6h": events["fire_age_hours"] <= 6,
        "le_12h": events["fire_age_hours"] <= 12,
        "le_24h": events["fire_age_hours"] <= 24,
        "24_48h": (events["fire_age_hours"] > 24) & (events["fire_age_hours"] <= 48),
    }.items():
        events[f"frp_{label}"] = events["frp"].where(mask)

    events["alignment_weighted_count"] = events["wind_alignment"]
    events["aligned_frp"] = events["frp"] * events["wind_alignment"]
    events["upwind_frp"] = events["frp"] * events["upwind_flag"]
    events["upwind_distance_km"] = events["distance_km"].where(events["upwind_flag"].eq(1.0))

    # Transparent FSI components.
    events["distance_weight"] = np.exp(-events["distance_km"] / FSI_DISTANCE_TAU_KM)
    age_lambda = np.log(2.0) / FSI_AGE_HALF_LIFE_HOURS
    events["age_weight"] = np.exp(-age_lambda * events["fire_age_hours"])
    events["wind_speed_factor"] = np.log1p(events["wind_speed_mean"].clip(lower=0))
    events["confidence_weight"] = confidence_weight(events["confidence"])
    events["frp_log"] = np.log1p(events["frp"].clip(lower=0))
    events["fsi_contribution"] = (
        events["frp_log"]
        * events["distance_weight"]
        * events["wind_alignment"]
        * events["wind_speed_factor"]
        * events["age_weight"]
        * events["confidence_weight"]
    )

    g = events.groupby("predictor_date")
    basic = g.agg(
        fire_count_48h=("latitude", "size"),
        fire_count_recent_6h=("count_le_6h", "sum"),
        fire_count_recent_12h=("count_le_12h", "sum"),
        fire_count_recent_24h=("count_le_24h", "sum"),
        fire_count_age_24_48h=("count_24_48h", "sum"),
        frp_sum_48h=("frp", lambda x: x.sum(min_count=1)),
        frp_sum_recent_6h=("frp_le_6h", lambda x: x.sum(min_count=1)),
        frp_sum_recent_12h=("frp_le_12h", lambda x: x.sum(min_count=1)),
        frp_sum_recent_24h=("frp_le_24h", lambda x: x.sum(min_count=1)),
        frp_sum_age_24_48h=("frp_24_48h", lambda x: x.sum(min_count=1)),
        fire_age_mean_hours=("fire_age_hours", "mean"),
        fire_age_min_hours=("fire_age_hours", "min"),
        alignment_mean=("wind_alignment", "mean"),
        alignment_weighted_fire_count=("alignment_weighted_count", lambda x: x.sum(min_count=1)),
        aligned_frp_sum=("aligned_frp", lambda x: x.sum(min_count=1)),
        upwind_fire_count=("upwind_flag", lambda x: x.sum(min_count=1)),
        upwind_frp_sum=("upwind_frp", lambda x: x.sum(min_count=1)),
        nearest_upwind_fire_km=("upwind_distance_km", "min"),
        fsi_total=("fsi_contribution", lambda x: x.sum(min_count=1)),
        fsi_max_single_fire=("fsi_contribution", "max"),
    ).reset_index()

    basic["upwind_frp_fraction"] = np.where(
        basic["frp_sum_48h"] > 0,
        basic["upwind_frp_sum"] / basic["frp_sum_48h"],
        0.0,
    )
    basic["station"] = station_name
    return basic.rename(columns={"predictor_date": "date_darwin"})

In [ ]:
advanced_station_fire = pd.concat(
    [
        build_advanced_fire_features_for_station(firms_fire, station, ntepa_daily)
        for station in station_coordinates
    ],
    ignore_index=True,
)

station_fire_complete = station_fire_complete.merge(
    advanced_station_fire,
    on=["date_darwin", "station"],
    how="left",
    validate="one_to_one",
)

advanced_zero_cols = [
    "fire_count_48h", "fire_count_recent_6h", "fire_count_recent_12h",
    "fire_count_recent_24h", "fire_count_age_24_48h", "frp_sum_48h",
    "frp_sum_recent_6h", "frp_sum_recent_12h", "frp_sum_recent_24h",
    "frp_sum_age_24_48h", "alignment_weighted_fire_count", "aligned_frp_sum",
    "upwind_fire_count", "upwind_frp_sum", "upwind_frp_fraction",
    "fsi_total", "fsi_max_single_fire",
]
complete_48h = station_fire_complete["firms_48h_complete"].eq(1)
for col in advanced_zero_cols:
    if col in station_fire_complete.columns:
        station_fire_complete.loc[complete_48h, col] = station_fire_complete.loc[complete_48h, col].fillna(0)
        station_fire_complete.loc[~complete_48h, col] = np.nan

advanced_nonzero_only = ["nearest_upwind_fire_km", "alignment_mean", "fire_age_mean_hours", "fire_age_min_hours"]
for col in advanced_nonzero_only:
    if col in station_fire_complete.columns:
        station_fire_complete.loc[~complete_48h, col] = np.nan

station_fire_complete.to_csv(processed_dir / "Fire2Air_firms_daily_checkpoint_A3.csv", index=False)

In [ ]:
fire_daily = station_fire_complete.rename(columns={"date_darwin": "date"})
integrated_daily = ntepa_daily.merge(
    fire_daily,
    on=["station", "date"],
    how="left",
    validate="one_to_one",
)

integrated_daily = integrated_daily.sort_values(["station", "date"]).reset_index(drop=True)
integrated_daily["month"] = integrated_daily["date"].dt.month
integrated_daily["day_of_year"] = integrated_daily["date"].dt.dayofyear
integrated_daily["month_sin"] = np.sin(2 * np.pi * integrated_daily["month"] / 12.0)
integrated_daily["month_cos"] = np.cos(2 * np.pi * integrated_daily["month"] / 12.0)
integrated_daily["doy_sin"] = np.sin(2 * np.pi * integrated_daily["day_of_year"] / 365.25)
integrated_daily["doy_cos"] = np.cos(2 * np.pi * integrated_daily["day_of_year"] / 365.25)
integrated_daily["wet_season"] = integrated_daily["month"].isin([11, 12, 1, 2, 3, 4]).astype(int)

# Air-quality persistence features available by the end of predictor day t.
grouped = integrated_daily.groupby("station", group_keys=False)
integrated_daily["pm25_lag1"] = grouped["pm25_mean"].shift(1)
integrated_daily["pm25_lag2"] = grouped["pm25_mean"].shift(2)
integrated_daily["pm25_lag3"] = grouped["pm25_mean"].shift(3)
integrated_daily["pm10_lag1"] = grouped["pm10_mean"].shift(1)
integrated_daily["pm25_change_1d"] = integrated_daily["pm25_mean"] - integrated_daily["pm25_lag1"]

integrated_daily["pm25_3day_mean"] = grouped["pm25_mean"].transform(
    lambda x: x.rolling(3, min_periods=3).mean()
)
integrated_daily["pm25_7day_mean"] = grouped["pm25_mean"].transform(
    lambda x: x.rolling(7, min_periods=5).mean()
)
integrated_daily["pm25_7day_max"] = grouped["pm25_mean"].transform(
    lambda x: x.rolling(7, min_periods=5).max()
)
integrated_daily["pm25_7day_std"] = grouped["pm25_mean"].transform(
    lambda x: x.rolling(7, min_periods=5).std()
)

# Meteorological lag/change features.
for source_col, prefix in [
    ("humidity_mean", "humidity"),
    ("temperature_mean", "temperature"),
    ("wind_speed_mean", "wind_speed"),
    ("pressure_mean", "pressure"),
    ("rainfall_total", "rainfall"),
]:
    integrated_daily[f"{prefix}_lag1"] = grouped[source_col].shift(1)
    integrated_daily[f"{prefix}_change_1d"] = (
        integrated_daily[source_col] - integrated_daily[f"{prefix}_lag1"]
    )

# Basic fire persistence complements the explicit 0–48 h age features.
integrated_daily["fire_count_0_500km_lag1"] = grouped["fire_count_0_500km"].shift(1)
integrated_daily["frp_sum_0_500km_lag1"] = grouped["frp_sum_0_500km"].shift(1)